# 08 — Trayectorias de posición relativa de ingreso (medición B)

Capa 2 de la medición B: por cada par de hogares vinculado por el panel
(`data/03_panel/pares_h1`/`pares_h4`, todas las eras), la variación de
`y_ae = ITF/unidades_ae` entre `t` y `t+h` (`src/trayectoria_relativa.py`)
-- `variacion_log_b` (objetivo principal, solo `y_ae>0` en ambas puntas),
variación simétrica de la posición relativa (admite `y_ae=0`), componente
común/propio (`variacion_comun_b`/`variacion_propia_b`) y `par_valido_b`.

**Lee** `data/02c_posicion_relativa/hogar/` (Capa 1) y
`data/03_panel/{pares_h1,pares_h4}` (ya existentes, sin reconstruir).
**Escribe** `data/06_trayectoria_relativa/{pares_h1,pares_h4}/` y
`data/meta/trayectoria_relativa_reporte_capa2.csv`.

**Cubre toda la serie 2003T3-2025T4** (a diferencia de
`06_trayectorias_distancia.ipynb`, que solo cubre 2016T2+) -- confirmado
que ningún par cruza el cambio de formato de `CODUSU` entre eras
(`regimen_ingreso != "mixto"` en el 100% de `data/03_panel`, chequeado abajo).
Sin `tendencia_previa`/`direccion`/`transicion` (específicos de la medición
A, no se improvisa un equivalente esta ronda).

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import pipeline_meta as pm
import panel
import trayectoria_relativa
from armonizacion import trimestres_publicados

POSICION_RELATIVA = REPO / "data" / "02c_posicion_relativa" / "hogar"
PANEL_DIR = REPO / "data" / "03_panel"
TRAYECTORIA = REPO / "data" / "06_trayectoria_relativa"
META = REPO / "data" / "meta"
COMMIT_GIT = pm.obtener_commit_git(REPO)

TRIMESTRES_RELATIVA = trimestres_publicados()
print(f"{len(TRIMESTRES_RELATIVA)} trimestres de origen candidatos (toda la serie)")

## Carga de la Capa 1 en memoria

Una sola lectura por trimestre -- se reusa tanto como origen `t` de un
horizonte como destino `t+h` de otro, sin releer del disco.

In [ ]:
posicion_por_trimestre = {}
for anio, trimestre in TRIMESTRES_RELATIVA:
    p = POSICION_RELATIVA / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    if p.exists():
        posicion_por_trimestre[(anio, trimestre)] = pd.read_parquet(p)
print(f"{len(posicion_por_trimestre)} particiones de posición relativa cargadas")

## Construcción de pares h1 y h4

Mismo helper para los dos horizontes: lee la partición de
`data/03_panel/pares_h{h}` ya construida (todas las eras), verifica que
ninguna fila sea `regimen_ingreso=="mixto"` (si lo fuera, un par estaría
cruzando el cambio de formato de `CODUSU` entre eras -- no debería pasar
nunca, por construcción de `panel.vincular_hogares`), y escribe
`data/06_trayectoria_relativa/pares_h{h}/`.

In [ ]:
def construir_capa2(h: int) -> tuple[dict, list]:
    pares_por_trimestre = {}
    reportes = []
    for anio, trimestre in TRIMESTRES_RELATIVA:
        anio_th, trimestre_th = panel.sumar_trimestres(anio, trimestre, h)
        if (anio, trimestre) not in posicion_por_trimestre or (anio_th, trimestre_th) not in posicion_por_trimestre:
            continue
        p_panel = pm.ruta_particion(PANEL_DIR, f"pares_h{h}", anio, trimestre)
        if not p_panel.exists():
            continue
        pares_hogar = pd.read_parquet(p_panel)
        if pares_hogar.empty:
            # 2013T2 y 2015T2 en pares_h4: ruptura puntual de CODUSU en esas
            # cohortes de entrada (ya documentada), 0 pares vinculados -- no
            # hay nada que computar, no es un error.
            continue
        n_mixto = (pares_hogar["regimen_ingreso"] == "mixto").sum()
        assert n_mixto == 0, f"pares_h{h} {anio}T{trimestre}: {n_mixto} pares cruzan el cambio de CODUSU"

        pares = trayectoria_relativa.construir_pares_relativa(
            posicion_por_trimestre[(anio, trimestre)], posicion_por_trimestre[(anio_th, trimestre_th)], pares_hogar)
        pares_por_trimestre[(anio, trimestre)] = pares

        destino = pm.ruta_particion(TRAYECTORIA, f"pares_h{h}", anio, trimestre)
        destino.parent.mkdir(parents=True, exist_ok=True)
        pares.to_parquet(destino, index=False)

        reporte = trayectoria_relativa.contar_exclusiones_y_ae(pares)
        reporte.update({"h": h, "ANO4_t": anio, "TRIMESTRE_t": trimestre})
        reportes.append(reporte)
    return pares_por_trimestre, reportes

In [ ]:
t0 = time.time()
pares_h1_por_trimestre, reportes_h1 = construir_capa2(1)
duracion_h1 = time.time() - t0
n_pares_h1 = sum(len(df) for df in pares_h1_por_trimestre.values())
print(f"pares_h1: {len(pares_h1_por_trimestre)} trimestres de origen, {n_pares_h1:,} pares, {duracion_h1:.1f}s")

t0 = time.time()
pares_h4_por_trimestre, reportes_h4 = construir_capa2(4)
duracion_h4 = time.time() - t0
n_pares_h4 = sum(len(df) for df in pares_h4_por_trimestre.values())
print(f"pares_h4: {len(pares_h4_por_trimestre)} trimestres de origen, {n_pares_h4:,} pares, {duracion_h4:.1f}s")

pm.registrar_corrida(META / "corridas.csv", "08_trayectoria_relativa", {"cobertura": "todas_las_eras"},
                      len(pares_h1_por_trimestre) + len(pares_h4_por_trimestre), n_pares_h1 + n_pares_h4,
                      duracion_h1 + duracion_h4, COMMIT_GIT)

## Reporte de exclusiones por `y_ae`

Cuántos pares quedan sin `variacion_log_b` (objetivo principal de B) y por
qué -- desagregado por horizonte y por era de origen.

In [ ]:
def _era_origen(pares_por_trimestre: dict, anio: int, trimestre: int) -> str:
    df = pares_por_trimestre[(anio, trimestre)]
    return df["era_t"].iloc[0] if len(df) else "sin_datos"


filas_reporte = []
for h, reportes, pares_por_trimestre in ((1, reportes_h1, pares_h1_por_trimestre), (4, reportes_h4, pares_h4_por_trimestre)):
    for r in reportes:
        r["era_t"] = _era_origen(pares_por_trimestre, r["ANO4_t"], r["TRIMESTRE_t"])
        filas_reporte.append(r)

reporte_capa2 = pd.DataFrame(filas_reporte)
resumen_por_era = reporte_capa2.groupby(["h", "era_t"])[
    ["n_pares", "n_sin_variacion_log_b", "n_y_ae_t_nulo", "n_y_ae_th_nulo",
     "n_y_ae_t_no_positivo", "n_y_ae_th_no_positivo"]
].sum()
resumen_por_era.to_csv(META / "trayectoria_relativa_reporte_capa2.csv")
print("data/meta/trayectoria_relativa_reporte_capa2.csv")
resumen_por_era